# 04 · Learning-rate check (D30) and corruption suite (H2a)

Runs on **Kaggle** as a saved version (Save Version → Save & Run All). No retraining of the core grid.

### Inputs to attach (Add Input)
1. The private cache dataset (same as notebook 03).
2. *Your Work* → the **output of the notebook-03 version that produced the grid**. It holds `work.tar` with every
   run's `ckpt_best.pt` and the SSL encoders. The results export zip is not enough: it has no checkpoints.

Settings: **Accelerator GPU T4**, **Internet on**.

### What it does
- **Part A, D30 check (exploratory):** fine-tunes C1 with max lr 1e-3 at the 5% and 100% budgets, seeds 0–2
  (6 runs, about 25 min). Runs go to `runs_lrcheck/` so the grid registry is untouched.
- **Part B, corruption suite:** evaluates the saved best checkpoints on the 9 held-out and 9 seen conditions.
  It refuses to run until the corruption protocol is frozen (`corrupt.FROZEN`, decision D35).

Set the two switches in the next cell. Download `results_export_04.zip` from the Output tab afterwards.

In [ ]:
RUN_LR_CHECK = True
RUN_CORRUPTIONS = True
ALLOW_INTERACTIVE = False     # True only for short tests you are willing to lose (lab notebook, Failure 7)

In [ ]:
import os, glob, json, subprocess, sys, tarfile, zipfile
if not os.path.isdir('/kaggle/working'):
    raise RuntimeError('This notebook is written for Kaggle.')
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != 'Batch' and not ALLOW_INTERACTIVE:
    raise RuntimeError('Interactive session: results would be lost. Use Save Version -> Save & Run All (Commit).')
REPO, CACHE, WORK = '/tmp/robust-ecg-ssl', '/tmp/cache', '/tmp/work'
EXPORT = '/kaggle/working/results_export_04.zip'
hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ptbxl_X.npy', recursive=True)]
if not hits:
    raise FileNotFoundError('No input dataset with ptbxl_X.npy: add the cache dataset.')
prior = sorted(glob.glob('/kaggle/input/**/work.tar', recursive=True))
if not prior:
    raise FileNotFoundError('No work.tar among the inputs: add the output of the notebook-03 grid version.')
if not os.path.exists(WORK):
    with tarfile.open(prior[-1]) as t:
        t.extractall('/tmp')
    print('restored', prior[-1])
n_ckpt = len(glob.glob(f'{WORK}/runs/*/ckpt_best.pt'))
n_enc = len(glob.glob(f'{WORK}/ssl/*/encoder.pt'))
print(n_ckpt, 'checkpoints,', n_enc, 'SSL encoders')
assert n_ckpt == 120 and n_enc == 5, 'expected the complete core grid (120 checkpoints, 5 encoders)'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Salako07/robust-ecg-ssl.git', REPO], check=True)
subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, f'{REPO}/src')
from robust_ecg.colab_utils import copy_dir_with_retry
copy_dir_with_retry(hits[0], CACHE, remount=None)
import torch
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Part A · D30 learning-rate check
C1 fine-tuned with max lr 1e-3 instead of 1e-2; everything else as in training protocol v1. Exploratory: it is
reported whatever it shows and does not replace the primary C1 results. Runs one after another on one GPU, because
parallel runs would race when appending to the registry (D32).

In [ ]:
LR_RUNS = f'{WORK}/runs_lrcheck'
if RUN_LR_CHECK:
    for seed in (0, 1, 2):
        for budget in (0.05, 1.0):
            cmd = [sys.executable, f'{REPO}/scripts/train.py', '--arm', 'C1', '--budget', str(budget), '--seed', str(seed),
                   '--cache', CACHE, '--runs', LR_RUNS, '--lr', '1e-3', '--tag', 'lr1e-3',
                   '--pretrained', f'{WORK}/ssl/SSL_ecg-mid_s{seed}/encoder.pt']
            r = subprocess.run(cmd, capture_output=True, text=True)
            print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-2000:])
            r.check_returncode()
    import pandas as pd
    lr = pd.read_csv(f'{LR_RUNS}/registry.csv')
    grid = pd.read_csv(f'{WORK}/runs/registry.csv')
    ref = grid[grid.arm.isin(['C1', 'S1']) & grid.seed.isin([0, 1, 2]) & grid.budget.isin([0.05, 1.0])]
    cols = ['test_macro_auroc', 'sph_e3_macro_auroc', 'e3_degradation', 'best_step']
    print('C1 at lr 1e-3:'); print(lr.set_index(['budget', 'seed'])[cols].round(4).sort_index().to_string())
    print('\nGrid (lr 1e-2), same seeds:'); print(ref.set_index(['arm', 'budget', 'seed'])[cols].round(4).sort_index().to_string())

## Part B · Corruption suite
Budget 100% first, with predictions saved for the patient bootstrap (primary contrast H2a). Then the other
budgets, metrics only (secondary).

In [ ]:
from robust_ecg import corrupt
if RUN_CORRUPTIONS and not corrupt.FROZEN:
    print('SKIPPED: corrupt.FROZEN is False, the corruption protocol (D35) is not frozen yet. Part A is still exported.')
elif RUN_CORRUPTIONS:
    base = [sys.executable, f'{REPO}/scripts/eval_corruptions.py', '--cache', CACHE, '--runs', f'{WORK}/runs']
    subprocess.run(base + ['--budgets', '1.0'], check=True)
    subprocess.run(base + ['--budgets', '0.01', '0.05', '0.1', '0.25', '0.5', '--no-preds'], check=True)
    import pandas as pd
    c = pd.read_csv(f'{WORK}/runs/corruptions_registry.csv')
    print(c.arm.value_counts().to_dict(), '| runs:', c.run_id.nunique())

## Export
Small files only. Model checkpoints stay in the archive below.

In [ ]:
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{WORK}/runs/*/corruptions.json') + glob.glob(f'{WORK}/runs/*/corruptions.npz') + \
             glob.glob(f'{WORK}/runs/corruptions_registry.csv') + glob.glob(f'{WORK}/runs_lrcheck/registry.csv'):
        z.write(f, os.path.relpath(f, WORK))
    for f in glob.glob(f'{WORK}/runs_lrcheck/*/*'):
        if f.endswith(('config.json', 'log.csv', 'done.json', 'predictions.npz')):
            z.write(f, os.path.relpath(f, WORK))
print(EXPORT, round(os.path.getsize(EXPORT) / 1e6, 2), 'MB,', len(zipfile.ZipFile(EXPORT).namelist()), 'files')
# keep the full state (checkpoints included) for later notebooks
with tarfile.open('/kaggle/working/work.tar.tmp', 'w') as t:
    t.add(WORK, arcname='work')
os.replace('/kaggle/working/work.tar.tmp', '/kaggle/working/work.tar')
print('work.tar', round(os.path.getsize('/kaggle/working/work.tar') / 1e6), 'MB')